# 01 — Model Training: aggregate instance segmentation

Baseline + ablation pipeline for `yolo26n-seg` on the cleaned datasets from `00EDA.ipynb`.

**Experiment matrix (minimal-first, approved):**

| phase | what | why |
|---|---|---|
| 0 | environment + dataset wiring | reproducibility audit |
| 1 | smoke test (overfit 24 imgs) | prove pipeline learns before burning GPU-hours |
| 2 | autobatch probe | batch=-1 -> max safe batch on this GPU |
| 3 | optimizer bake-off: MuSGD vs AdamW vs SGD | official YOLO26 recipe uses MuSGD; decide empirically |
| 4 | data arms: `gold` vs `gold_aug` | does offline expansion (incl. dark/rain) actually help |
| 5 | holdout `test` — evaluated **once**, on the winner only | untouched final claim |

**Auto rules wired in:**
- early stop: `patience` on val fitness (0.1*mAP50 + 0.9*mAP50-95, mask)
- auto-LR retry: if a run diverges or plateaus at a clearly-broken level -> retry once at lr0/2 and keep the better run
- auto-batch: `check_train_batch_size` probe before any real run

**Periodic viz:** a fixed panel of validation images is predicted every `VIZ_EVERY`
epochs during each real run (callback -> PNG under `runs/<name>/viz/`).

Requirements: `ultralytics>=8.4.146`, torch with CUDA/ROCm (`pip install ultralytics`).
Datasets expected at `data/gold`, `data/gold_aug` (optionally `data/gold_clahe`) —
on a fresh machine either copy `experiment/data/` from the release asset or re-run `00EDA.ipynb`.

## 0. Environment audit

VIZ/MEASURE cell — prints what the run will actually see. No mutations.

In [ ]:
# ============================================================
# 0.1 ENV CHECK — torch / GPU / ultralytics / seed
# ============================================================
import os, sys, json, shutil, random, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
import torch
import ultralytics
from ultralytics import YOLO
from PIL import Image

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

EXP_DIR  = Path.cwd()
DATA_DIR = EXP_DIR / "data"
RUNS_DIR = EXP_DIR / "runs"
RES_DIR  = EXP_DIR / "results"
RUNS_DIR.mkdir(exist_ok=True); RES_DIR.mkdir(exist_ok=True)

HAS_GPU = torch.cuda.is_available()
DEVICE  = "0" if HAS_GPU else "cpu"
WORKERS = 4 if os.name != "nt" else 0   # Windows spawn-safety

print(f"python      : {sys.version.split()[0]}")
print(f"ultralytics : {ultralytics.__version__}")
print(f"torch       : {torch.__version__}  (hip={torch.version.hip})")
print(f"gpu         : {HAS_GPU}")
if HAS_GPU:
    print(f"device      : {torch.cuda.get_device_name(0)}")
    print(f"vram        : {torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GB")
print(f"exp_dir     : {EXP_DIR}")

In [ ]:
# ============================================================
# 0.2 DATASET WIRING — resolve dirs, write absolute data.yamls
# ============================================================
# gold/data.yaml ships with Roboflow-style relative paths; we write
# our own per-dataset yaml with ABSOLUTE paths so runs are portable.

DATASETS = {}                       # name -> dataset root
for name in ("gold", "gold_aug", "gold_clahe"):
    for cand in (DATA_DIR / name, Path.home() / "thesis_data" / name):
        if (cand / "train" / "images").is_dir():
            DATASETS[name] = cand.resolve()
            break

assert "gold" in DATASETS and "gold_aug" in DATASETS, \
    "need data/gold and data/gold_aug — copy experiment/data/ here or re-run 00EDA.ipynb"

CFG_DIR = RUNS_DIR / "_cfg"; CFG_DIR.mkdir(exist_ok=True)
DATA_YAML = {}
for name, root in DATASETS.items():
    y = {"path": str(root),
         "train": "train/images", "val": "valid/images", "test": "test/images",
         "nc": 2, "names": ["gravel", "sand"]}
    p = CFG_DIR / f"{name}.yaml"
    p.write_text(yaml.safe_dump(y))
    DATA_YAML[name] = str(p)
    counts = {s: len(list((root / s / "images").iterdir())) for s in ("train", "valid", "test")}
    print(f"{name:12s} -> {counts}  yaml={p.name}")

## 1. Shared machinery

Helpers every phase uses: aug policy from `train.yaml`, fixed val panel,
periodic-viz callback, results parsing, and `train_with_retry` (auto-LR).

In [ ]:
# ============================================================
# 1.1 HELPERS — aug policy, results parsing, fitness
# ============================================================

# ---- aug policy: single source of truth is train.yaml --------
AUG_KEYS = ["mosaic", "close_mosaic", "copy_paste", "fliplr", "flipud",
            "degrees", "scale", "translate", "perspective", "shear",
            "hsv_h", "hsv_s", "hsv_v", "erasing"]
_train_yaml = yaml.safe_load((EXP_DIR / "train.yaml").read_text())
AUG = {k: _train_yaml[k] for k in AUG_KEYS if k in _train_yaml}
print("aug policy:", AUG)


def read_results(run_dir):
    """ultralytics results.csv -> DataFrame (column names stripped)."""
    df = pd.read_csv(Path(run_dir) / "results.csv")
    df.columns = [c.strip() for c in df.columns]
    return df


def map_col(df):
    """mask mAP50-95 column name, version-tolerant."""
    for c in df.columns:
        if "map50-95(m" in c.lower():
            return c
    for c in df.columns:
        if "map50-95" in c.lower():
            return c
    raise KeyError(f"no mAP50-95 col in {list(df.columns)}")


def run_fitness(run_dir):
    """fitness = 0.1*mask mAP50 + 0.9*mask mAP50-95 (ultralytics convention)."""
    df = read_results(run_dir)
    m50  = df[[c for c in df.columns if "map50(" in c.lower() and "95" not in c.lower()][-1]]
    m95  = df[map_col(df)]
    fit  = 0.1 * m50 + 0.9 * m95
    return df, fit

In [ ]:
# ============================================================
# 1.2 FIXED VIZ PANEL + PERIODIC-VIZ CALLBACK + TRAIN WRAPPER
# ============================================================

# ---- panel: 6 fixed valid images — flagged-first if flags exist
val_imgs = sorted((DATASETS["gold"] / "valid" / "images").iterdir())
qf_path  = DATASETS["gold"] / "quality_flags.csv"
panel = []
if qf_path.exists():
    qf = pd.read_csv(qf_path)
    qv = qf[qf.get("split", pd.Series(["valid"]*len(qf))).astype(str).str.contains("valid")] \
           if "split" in qf.columns else qf
    flag_cols = [c for c in qf.columns
                 if ("flag" in c.lower() or c.startswith("is_"))
                 and qf[c].dtype != object]
    if flag_cols:
        flagged = qv[qv[flag_cols].any(axis=1)]["image"].tolist() if "image" in qv.columns else []
        panel = flagged[:3]
panel += [p.name for p in val_imgs if p.name not in panel][:6 - len(panel)]
PANEL = [str(DATASETS["gold"] / "valid" / "images" / n) for n in panel]
print("viz panel:", [Path(p).name[:40] for p in PANEL])

VIZ_EVERY = 10

def viz_cb(trainer):
    """every VIZ_EVERY epochs (and last): predict panel -> runs/<name>/viz/."""
    ep = trainer.epoch + 1
    if ep % VIZ_EVERY and ep != trainer.epochs:
        return
    try:
        out = Path(trainer.save_dir) / "viz"
        out.mkdir(exist_ok=True)
        y = YOLO(trainer.model)
        rs = y.predict(PANEL, imgsz=640, device=trainer.device, verbose=False)
        fig, axes = plt.subplots(1, len(rs), figsize=(3 * len(rs), 3))
        for ax, r in zip(np.atleast_1d(axes), rs):
            ax.imshow(r.plot()[..., ::-1]); ax.axis("off")
        fig.savefig(out / f"ep{ep:03d}.png", dpi=90, bbox_inches="tight")
        plt.close(fig)
    except Exception as e:                       # viz must never kill training
        print(f"[viz_cb] ep{ep} skipped: {e}")


def train_run(name, data, model_src="yolo26n-seg.pt", epochs=100, patience=30,
              batch=16, optimizer="AdamW", lr0=1e-3, lrf=0.01, momentum=0.937,
              weight_decay=5e-4, warmup_epochs=3.0, viz=True, **kw):
    """one training run. returns dict(record)."""
    t0 = time.time()
    model = YOLO(model_src)
    if viz:
        model.add_callback("on_train_epoch_end", viz_cb)
    args = dict(data=data, task="segment", epochs=epochs, patience=patience,
                batch=batch, imgsz=640, device=DEVICE, workers=WORKERS,
                optimizer=optimizer, lr0=lr0, lrf=lrf, momentum=momentum,
                weight_decay=weight_decay, warmup_epochs=warmup_epochs,
                cos_lr=True, amp=True, seed=SEED, deterministic=True,
                project=str(RUNS_DIR), name=name, exist_ok=False,
                plots=True, save_period=-1, verbose=False,
                **AUG, **kw)
    model.train(**args)
    save_dir = Path(model.trainer.save_dir)
    df, fit = run_fitness(save_dir)
    rec = {"name": name, "data": Path(str(data)).stem, "model": str(model_src),
           "optimizer": optimizer, "lr0": lr0, "batch": batch,
           "epochs_run": len(df), "best_fit": float(fit.max()),
           "best_epoch": int(fit.idxmax()) + 1,
           "final_fit": float(fit.iloc[-1]),
           "wall_min": round((time.time() - t0) / 60, 1),
           "save_dir": str(save_dir)}
    return rec


def needs_lr_retry(rec):
    """auto-LR trigger: diverged (fit ~0 / NaN) or hard-stalled low."""
    if not np.isfinite(rec["best_fit"]) or rec["best_fit"] < 0.02:
        return True
    df, fit = run_fitness(rec["save_dir"])
    tail = fit.iloc[max(0, len(fit) - 10):]      # last <=10 epochs
    return bool(len(fit) >= 10 and rec["best_fit"] < 0.15
                and (tail.max() - tail.min()) < 0.005)


def train_with_retry(name, **kw):
    """train; if needs_lr_retry -> one retry at lr0/2, keep the better."""
    rec = train_run(name, **kw)
    if needs_lr_retry(rec) and kw.get("lr0", 1e-3) > 1e-5:
        print(f"[{name}] weak run (best_fit={rec['best_fit']:.3f}) "
              f"-> retry lr0 {kw['lr0']} -> {kw['lr0']/2:g}")
        rec2 = train_run(name + "_lrhalf", **{**kw, "lr0": kw["lr0"] / 2})
        if rec2["best_fit"] > rec["best_fit"]:
            rec = rec2
    return rec

## 2. Smoke test — can the pipeline learn at all?

Overfit a 24-image subset for 10 epochs with augmentation OFF. If mask mAP
does not climb toward ~1.0, something upstream (labels, yaml, head) is broken
and no real run should start.

VIZ first: training curves + predictions on the same 24 images.

In [ ]:
# ============================================================
# 2.1 BUILD SMOKE SUBSET — 24 train images, self-contained
# ============================================================
SMOKE = RUNS_DIR / "_smoke"
if SMOKE.exists():
    shutil.rmtree(SMOKE)
for sub in ("train/images", "train/labels", "valid/images", "valid/labels"):
    (SMOKE / sub).mkdir(parents=True)

rng = np.random.default_rng(SEED)
tr_imgs = sorted((DATASETS["gold"] / "train" / "images").iterdir())
pick = rng.choice(len(tr_imgs), size=24, replace=False)
for i in pick:
    p = tr_imgs[i]
    for s in ("train", "valid"):                    # same imgs -> prove overfit
        shutil.copy2(p, SMOKE / s / "images" / p.name)
        lt = DATASETS["gold"] / "train" / "labels" / (p.stem + ".txt")
        shutil.copy2(lt, SMOKE / s / "labels" / lt.name)

(SMOKE / "data.yaml").write_text(yaml.safe_dump({
    "path": str(SMOKE.resolve()), "train": "train/images", "val": "valid/images",
    "nc": 2, "names": ["gravel", "sand"]}))
print("smoke subset:", len(list((SMOKE/'train/images').iterdir())), "images")

In [ ]:
# ============================================================
# 2.2 OVERFIT RUN — 10 epochs, aug OFF, then viz curves + preds
# ============================================================
NO_AUG = dict(mosaic=0.0, fliplr=0.0, flipud=0.0, degrees=0.0, scale=0.0,
              translate=0.0, perspective=0.0, shear=0.0,
              hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, erasing=0.0, copy_paste=0.0)

smoke = train_run("smoke", str(SMOKE / "data.yaml"), epochs=10, patience=0,
                  batch=8, optimizer="AdamW", lr0=1e-3, viz=False, **NO_AUG)
print(smoke)

df, fit = run_fitness(smoke["save_dir"])
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].plot(df["epoch"], fit, marker="o"); ax[0].set_title("mask fitness (smoke)")
loss_cols = [c for c in df.columns if "loss" in c and "train" in c.lower()]
for c in loss_cols[:4]:
    ax[1].plot(df["epoch"], df[c], label=c.split("/")[-1])
    ax[1].set_title("train losses"); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.show(); plt.close("all")

bm = YOLO(Path(smoke["save_dir"]) / "weights" / "best.pt")
rs = bm.predict([str(SMOKE/'valid/images'/n) for n in
                 sorted(os.listdir(SMOKE/'valid/images'))[:6]],
                imgsz=640, device=DEVICE, verbose=False)
fig, axes = plt.subplots(1, len(rs), figsize=(3*len(rs), 3))
for ax, r in zip(np.atleast_1d(axes), rs):
    ax.imshow(r.plot()[..., ::-1]); ax.axis("off")
plt.suptitle("smoke preds (should nearly memorise the 24 imgs)")
plt.show(); plt.close("all")

## 3. AutoBatch probe

MEASURE cell — profiles the seg model on this GPU and picks the largest safe
batch (~60% VRAM target). The number is locked into `BATCH` for all later runs
so every arm trains under identical conditions.

In [ ]:
# ============================================================
# 3.1 AUTOBATCH — check_train_batch_size(model, fraction=0.6)
# ============================================================
from ultralytics.utils.autobatch import check_train_batch_size

if HAS_GPU:
    probe = YOLO("yolo26n-seg.yaml").model            # arch only, random init ok
    BATCH = check_train_batch_size(probe, imgsz=640, amp=True, batch=0.6)
    del probe; torch.cuda.empty_cache()
else:
    BATCH = 8
    print("no GPU — forcing batch=8 (cpu smoke only)")
print(f"BATCH = {BATCH}  (~60% VRAM target)")

## 4. Optimizer bake-off — MuSGD vs AdamW vs SGD

Reference points (official Ultralytics YOLO26 recipe, docs/en/guides/yolo26-training-recipe):
- COCO-trained YOLO26 uses **MuSGD**; n-size recipe: `lr0=0.0054, momentum=0.947, wd=0.00064, warmup~1ep`
- `optimizer='auto'` picks MuSGD only above ~10k iterations — our dataset is small
  enough that auto would silently pick AdamW, so we decide explicitly.
- AdamW fine-tune convention: `lr0=1e-3`; SGD convention: `lr0=1e-2`.

Protocol: 30 epochs on `gold_aug` (the strongest candidate data arm), identical
seed/batch/aug, `patience=10`, `cos_lr=True`. Auto-LR retry still armed.
Winner = highest best mask fitness; tie -> fewer epochs to best.

In [ ]:
# ============================================================
# 4.1 BAKE-OFF — 3 optimizers x 30 epochs on gold_aug
# ============================================================
OPT_GRID = [
    dict(optimizer="MuSGD", lr0=5.4e-3, momentum=0.947, weight_decay=6.4e-4, warmup_epochs=1.0),
    dict(optimizer="AdamW", lr0=1.0e-3, momentum=0.937, weight_decay=5.0e-4, warmup_epochs=3.0),
    dict(optimizer="SGD",   lr0=1.0e-2, momentum=0.937, weight_decay=5.0e-4, warmup_epochs=3.0),
]

opt_recs = []
for cfg in OPT_GRID:
    rec = train_with_retry(f"opt_{cfg['optimizer']}",
                           data=DATA_YAML["gold_aug"],
                           epochs=30, patience=10, batch=BATCH,
                           lrf=0.01, viz=True, **cfg)
    opt_recs.append(rec)
    print(rec)

opt_df = pd.DataFrame(opt_recs)
opt_df.to_csv(RES_DIR / "optimizer_bakeoff.csv", index=False)
opt_df

In [ ]:
# ============================================================
# 4.2 VIZ — per-optimizer fitness curves + loss curves
# ============================================================
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for rec in opt_recs:
    df, fit = run_fitness(rec["save_dir"])
    lbl = f"{rec['optimizer']} (lr0={rec['lr0']:g})"
    ax[0].plot(df["epoch"], fit, label=lbl)
    if "val/seg_loss" in df.columns:
        ax[1].plot(df["epoch"], df["val/seg_loss"], label=lbl)
ax[0].set_title("mask fitness / epoch"); ax[0].legend(fontsize=8)
ax[1].set_title("val seg_loss / epoch"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show(); plt.close("all")

print(opt_df[["name","optimizer","lr0","best_fit","best_epoch","wall_min"]]
      .sort_values("best_fit", ascending=False).to_string(index=False))

### 4.3 DECISION — winner optimizer

Selection is automatic but logged: max `best_fit`; ties broken by earlier
`best_epoch` (faster convergence). Whatever wins is frozen into `WIN_OPT`
for the data-arm phase — do not hand-edit after looking at val.

In [ ]:
best_row = opt_df.sort_values(["best_fit", "best_epoch"],
                              ascending=[False, True]).iloc[0]
WIN_OPT = next(c for c in OPT_GRID if c["optimizer"] == best_row["optimizer"])
WIN_OPT = dict(WIN_OPT, lr0=float(best_row["lr0"]))   # keep retried lr if it won
print("WIN_OPT =", WIN_OPT)

## 5. Data arms — `gold` vs `gold_aug`

Does offline expansion (4x train, incl. dark/rain env variants) actually beat
raw gold? Same winner optimizer, same batch/seed, `epochs=100, patience=30`.
Periodic panel-viz PNGs land under `runs/arm_*/viz/` every 10 epochs.

In [ ]:
# ============================================================
# 5.1 ARM RUNS — winner optimizer on gold and gold_aug
# ============================================================
ARMS = ["gold", "gold_aug"]
if "gold_clahe" in DATA_YAML:
    ARMS.append("gold_clahe")

arm_recs = []
for arm in ARMS:
    rec = train_with_retry(f"arm_{arm}", data=DATA_YAML[arm],
                           epochs=100, patience=30, batch=BATCH,
                           lrf=0.01, viz=True, **WIN_OPT)
    arm_recs.append(rec)
    print(rec)

arm_df = pd.DataFrame(arm_recs)
arm_df.to_csv(RES_DIR / "data_arms.csv", index=False)
arm_df

In [ ]:
# ============================================================
# 5.2 VIZ — fitness curves, per-class mask AP, pred panels
# ============================================================
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
per_class = {}
for rec in arm_recs:
    df, fit = run_fitness(rec["save_dir"])
    ax[0].plot(df["epoch"], fit, label=rec["name"])
    bm = YOLO(Path(rec["save_dir"]) / "weights" / "best.pt")
    m = bm.val(data=DATA_YAML[rec["name"].replace("arm_", "")],
               split="val", imgsz=640, device=DEVICE, verbose=False, plots=False)
    try:
        per_class[rec["name"]] = list(m.seg.maps)          # per-class mAP50-95
    except Exception:
        per_class[rec["name"]] = [float(m.seg.map)] * 2

ax[0].set_title("mask fitness / epoch"); ax[0].legend(fontsize=8)
names = ["gravel", "sand"]
x = np.arange(len(names)); w = 0.8 / len(arm_recs)
for j, (arm, aps) in enumerate(per_class.items()):
    ax[1].bar(x + j * w, aps, w, label=arm)
ax[1].set_xticks(x + w * (len(arm_recs) - 1) / 2); ax[1].set_xticklabels(names)
ax[1].set_title("per-class mask mAP50-95 (val)"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show(); plt.close("all")

print(arm_df[["name","optimizer","lr0","best_fit","best_epoch","wall_min"]]
      .sort_values("best_fit", ascending=False).to_string(index=False))

In [ ]:
# ============================================================
# 5.3 VIZ — best.pt predictions on the fixed panel, per arm
# ============================================================
for rec in arm_recs:
    bm = YOLO(Path(rec["save_dir"]) / "weights" / "best.pt")
    rs = bm.predict(PANEL, imgsz=640, device=DEVICE, verbose=False)
    fig, axes = plt.subplots(1, len(rs), figsize=(3*len(rs), 3))
    for ax_, r in zip(np.atleast_1d(axes), rs):
        ax_.imshow(r.plot()[..., ::-1]); ax_.axis("off")
    fig.suptitle(f"{rec['name']} — fixed panel preds", fontsize=11)
    plt.show(); plt.close("all")

### 5.4 DECISION — winning data arm

Winner = highest `best_fit` on **valid**. The loser stays on disk for the
appendix; the winner alone proceeds to the holdout cell.

In [ ]:
win_arm = arm_df.sort_values(["best_fit", "best_epoch"],
                            ascending=[False, True]).iloc[0]
WIN_ARM = win_arm["name"].replace("arm_", "")
BEST_PT = Path(win_arm["save_dir"]) / "weights" / "best.pt"
print(f"winner: {WIN_ARM}  best_fit={win_arm['best_fit']:.4f}  ckpt={BEST_PT}")

## 6. Holdout evaluation — runs exactly once

This is the only cell that touches `test/`. Run it once, after every
selection above is locked. Anything after this is reporting, not tuning.

In [ ]:
# ============================================================
# 6.1 FINAL EVAL — winner arm on untouched test split
# ============================================================
best_model = YOLO(str(BEST_PT))
tm = best_model.val(data=DATA_YAML[WIN_ARM], split="test",
                    imgsz=640, batch=BATCH, device=DEVICE,
                    workers=WORKERS, plots=True, verbose=False)

final = {
    "arm": WIN_ARM, "optimizer": WIN_OPT["optimizer"], "lr0": WIN_OPT["lr0"],
    "batch": BATCH, "seed": SEED,
    "test_box_map5095": float(tm.box.map),  "test_box_map50": float(tm.box.map50),
    "test_box_p": float(tm.box.mp),         "test_box_r": float(tm.box.mr),
    "test_mask_map5095": float(tm.seg.map), "test_mask_map50": float(tm.seg.map50),
    "test_mask_p": float(tm.seg.mp),        "test_mask_r": float(tm.seg.mr),
}
try:
    for cls_i, ap in enumerate(tm.seg.maps):
        final[f"test_mask_map5095_{tm.names[cls_i]}"] = float(ap)
except Exception:
    pass
print(pd.Series(final))

with open(RES_DIR / "holdout_final.json", "w") as f:
    json.dump(final, f, indent=2)

In [ ]:
# ============================================================
# 6.2 VIZ — winner predictions on a fixed test panel
# ============================================================
test_imgs = sorted((DATASETS[WIN_ARM] / "test" / "images").iterdir())
tp = [str(p) for p in test_imgs[:6]]
rs = best_model.predict(tp, imgsz=640, device=DEVICE, verbose=False)
fig, axes = plt.subplots(1, len(rs), figsize=(3*len(rs), 3))
for ax_, r in zip(np.atleast_1d(axes), rs):
    ax_.imshow(r.plot()[..., ::-1]); ax_.axis("off")
plt.suptitle(f"{WIN_ARM} — test panel preds")
plt.show(); plt.close("all")

## 7. Summary + next steps

All run records -> `results/runs_summary.csv`; holdout -> `results/holdout_final.json`.

**Deliberately deferred (do not add silently):**
- P2 head (`yolo26n-seg-p2.yaml` + Segment26P2 patch) — next ablation round
- per-slice eval on `quality_flags.csv` (blur/dark/cast slices)
- round-2 label audit (model-vs-label disagreement -> human re-review)
- statistical paired tests — need >1 seed per arm; single-seed result is a point estimate

In [ ]:
summary = pd.concat([opt_df.assign(phase="optimizer"),
                     arm_df.assign(phase="data_arm")], ignore_index=True)
summary["winner_opt"] = WIN_OPT["optimizer"]
summary["winner_arm"] = WIN_ARM
summary.to_csv(RES_DIR / "runs_summary.csv", index=False)
summary